In [142]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,accuracy_score,classification_report
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas

ASSIGMENT 2 Neural Network Dataset 2

In [143]:
# CHOOSE DATASET

# Binary classification dataset
#data = datasets.load_breast_cancer(as_frame=True)

# Regression dataset
diabetes = fetch_openml("diabetes",version=1, as_frame=True) 

X = diabetes.data.astype(float).to_numpy()
y_origin = diabetes.target.values
X.shape

# Map string labels to integers: negative=0, positive=1
y = np.where(y_origin == "tested_positive", 1, 0).astype(int)

This code runs exactly the same as the NN for dataset 1 with just the change in dataset input.

In [144]:
#train test spliting
test_size=0.2
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, random_state=42)

In [145]:
# Standardize features
scaler=StandardScaler()
Xtr= scaler.fit_transform(Xtr)
Xte= scaler.transform(Xte)

In [146]:
class MLP(nn.Module):
    def __init__(self, input_size, output_size=1, dropout_prob=0.5):
        super(MLP, self).__init__()
        
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 64)
        self.fc4 = nn.Linear(64, 64)
        self.out = nn.Linear(64, output_size)
        
        self.dropout = nn.Dropout(p=dropout_prob)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        
        x = F.relu(self.fc4(x))
        x = self.dropout(x)
        
        x = self.out(x)
        return x

In [147]:
num_epochs=70
lr=0.0005
dropout=0.1
batch_size=64

In [148]:
Xtr = torch.tensor(Xtr, dtype=torch.float32)
ytr = torch.tensor(ytr, dtype=torch.float32)
Xte = torch.tensor(Xte, dtype=torch.float32)
yte = torch.tensor(yte, dtype=torch.float32)

# Wrap Xtr and ytr into a dataset
train_dataset = TensorDataset(Xtr, ytr)

# Create DataLoader
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [149]:
# Model, Loss, Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MLP(input_size=Xtr.shape[1], dropout_prob=dropout).to(device)
criterion = nn.BCEWithLogitsLoss()  # for binary classification
criterion = nn.MSELoss() #for regression
optimizer = optim.Adam(model.parameters(), lr=lr)

# Send data to device
Xtr, ytr = Xtr.to(device), ytr.to(device)
Xte, yte = Xte.to(device), yte.to(device)

In [150]:
# Training loop
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for batch_x, batch_y in train_dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        loss = criterion(logits, batch_y.view(-1, 1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

Epoch [1/70], Loss: 0.2555
Epoch [2/70], Loss: 0.2297
Epoch [3/70], Loss: 0.2081
Epoch [4/70], Loss: 0.1987
Epoch [5/70], Loss: 0.1849
Epoch [6/70], Loss: 0.1730
Epoch [7/70], Loss: 0.1637
Epoch [8/70], Loss: 0.1603
Epoch [9/70], Loss: 0.1589
Epoch [10/70], Loss: 0.1591
Epoch [11/70], Loss: 0.1577
Epoch [12/70], Loss: 0.1509
Epoch [13/70], Loss: 0.1509
Epoch [14/70], Loss: 0.1486
Epoch [15/70], Loss: 0.1532
Epoch [16/70], Loss: 0.1440
Epoch [17/70], Loss: 0.1520
Epoch [18/70], Loss: 0.1467
Epoch [19/70], Loss: 0.1407
Epoch [20/70], Loss: 0.1428
Epoch [21/70], Loss: 0.1452
Epoch [22/70], Loss: 0.1441
Epoch [23/70], Loss: 0.1392
Epoch [24/70], Loss: 0.1358
Epoch [25/70], Loss: 0.1411
Epoch [26/70], Loss: 0.1412
Epoch [27/70], Loss: 0.1350
Epoch [28/70], Loss: 0.1381
Epoch [29/70], Loss: 0.1349
Epoch [30/70], Loss: 0.1301
Epoch [31/70], Loss: 0.1347
Epoch [32/70], Loss: 0.1324
Epoch [33/70], Loss: 0.1330
Epoch [34/70], Loss: 0.1341
Epoch [35/70], Loss: 0.1334
Epoch [36/70], Loss: 0.1297
E

In [151]:
y_pred=model(Xte)

# move both to cpu and convert to numpy
y_true_np = yte.detach().cpu().numpy()
y_pred_np = y_pred.detach().cpu().numpy()

print(f'ACC:{accuracy_score(y_true_np, y_pred_np>0.5)}') #classification
#print(f'MSE:{mean_squared_error(yte.detach().numpy(),y_pred.detach().numpy())}') #regression

ACC:0.7532467532467533


In this case the Anfis has ance again better results. I do believe that it is due to the size of our dataset, because NN is build for more complex models.

With NN for network of this size, the result depends on the random disconnection of neurons. In many cases of the same parametrs the final accurace my differ in up to 4%.